# exostab classroom exercise: three answers to one question

[exostab](https://github.com/mimitoz62/exostab) asks in three ways whether a known multi-planet system is dynamically
stable: Hill spacing, the angular momentum deficit (AMD) and N-body integration with the chaos indicator MEGNO. In this
exercise you compare the three answers for real systems, and you explain what a short integration cannot show.

**Level and time.** Upper-undergraduate dynamics or exoplanets. Plan about 90 minutes. That is an estimate, because the
exercise has not yet been tried with a class. The cells compute for about 11 minutes of that, more on a slow computer;
you can read the next questions while a cell runs.

**What you need.** A browser and a Google account; nothing is installed on your computer. You should know Kepler's third
law, the orbital elements a, e and ω, and what chaos means: nearby trajectories that separate exponentially.

**How to work.**

1. Use **File → Save a copy in Drive** first and go on in the copy. Otherwise your answers are lost when the tab closes.
2. Run the cells one at a time, in order, with Shift+Enter. Colab asks you to sign in and to confirm that you want to run
   a notebook from GitHub. You can ignore the note of the first code cell about restarting the kernel. The results go to
   the folder `exostab-results/` and are deleted when the session ends. If Colab disconnects, use **Runtime → Run all**.
3. Read what a cell shows. Then write your answers in the text cell under the questions (double-click it), in your own
   words and with the numbers you use.
4. Hand in the notebook with its outputs and your answers: **File → Download → Download .ipynb**.

**The verdicts.** Each indicator says one of four things, for the published values (the *nominal* parameters) and for
random *draws* within the published error bars:

- **stable**: its condition holds for the nominal parameters and for at least 95% of the draws. This is not a proof. It
  means that exostab calls the system stable under these settings.
- **marginal**: borderline, or the draws disagree with the nominal parameters. This means mixed evidence, not "about to
  fall apart".
- **unstable**: its test fails for the nominal parameters and for at least 95% of the draws. For N-body, failing means that
  the run ended in a close encounter or escape.
- **inconclusive**: it could not be evaluated reliably.

The overall verdict is conservative. It is *stable* only if every indicator says stable, and *unstable* only if the
integration itself shows close encounters or escapes. It is *inconclusive* if the integrations could not be trusted, and
*marginal* in every other case, including any disagreement between the indicators.

One N-body run is *regular* if it has no close encounter or escape and ends with MEGNO ⟨Y⟩ below 2.2, and *chaotic* if it
has neither but ends with ⟨Y⟩ ≥ 2.2. [Methods](https://mimitoz62.github.io/exostab/methods/) explains the indicators.

**The inputs.** Every run uses the snapshot of 22 systems that ships with exostab (`--offline`, archive data of
28 September 2026) and seed 0, unless the cell says otherwise. A run labelled **what-if** changes published values by
hand: it describes that parameter set, not the real planets. Chaotic runs can differ between computers, so the exact ⟨Y⟩
and a draw near the threshold of 2.2 can differ from a classmate's on another computer. That is not a mistake.

**Run times.** The first line of each slow cell says how long it should take in Colab. These times were measured on one
core of an Apple M4 Pro laptop and multiplied by 4, because Colab's processors are expected to be slower. They have not
been measured in Colab itself.

In [ ]:
# Installs exostab from GitHub. This can take about a minute.
%pip install --quiet "exostab @ git+https://github.com/mimitoz62/exostab@v1.0.1"

In [ ]:
import copy
import csv
import json
import os

import yaml
from IPython.display import Image, display

# Plain text in the notebook: no progress spinner, no colours and, in Colab, lines of up to 110 characters.
os.environ.update(TERM="dumb", COLUMNS="110", LINES="50")
os.makedirs("exostab-results", exist_ok=True)

!exostab --version

## Task (a): a compact resonant system (about 20 minutes)

TRAPPIST-1 has seven planets within about 0.06 AU of a star of 0.09 solar masses. Two neighbouring planets are compared
by their separation in *mutual Hill radii*,

Δ = (a₂ − a₁) / R_H,  with R_H = ((m₁ + m₂) / 3M★)^(1/3) (a₁ + a₂) / 2.

Two planets on circular orbits can never come close to each other if Δ is larger than about 2√3 ≈ 3.46 (Gladman 1993).
For three or more planets there is no such proof, so exostab uses a rule from numerical experiments: Pu & Wu (2015) found
that Kepler-like systems need Δ ≈ 10 for half of them to survive a billion years with circular orbits, extrapolated from
shorter runs, and more when the orbits are eccentric. exostab's Hill indicator says *stable* for such a system only if
every neighbouring pair has Δ ≥ 12, more if the orbits are eccentric, and no eccentricity is above 0.05. It says
*unstable* if a pair fails the two-planet limit or has crossing orbits, and *marginal* otherwise.

When two periods are close to a simple fraction (a *mean-motion resonance*), the planets pass each other again and again
at about the same places on their orbits, so their pulls on each other can build up instead of averaging out. With
suitable orbital phases, this keeps the passes away from the points where the orbits are closest. Δ contains no phases,
so it cannot see this.

The *angular momentum deficit* is the angular momentum the orbits lack compared with circular orbits in one plane with
the same a:

C = Σₖ mₖ √(G M★ aₖ) (1 − √(1 − eₖ²) cos iₖ).

In the orbit-averaged (*secular*) dynamics, which describes slow changes over long times, the planets exchange AMD but
the total stays constant. β is C divided by the smallest AMD that lets a neighbouring pair cross or, for close pairs, lets
mean-motion resonances overlap. If β < 1 for every pair, that cannot happen in this approximation. Mutual inclinations are
almost never measured, so exostab assumes that they carry as much AMD as the eccentricities, which doubles the AMD
computed from the eccentricities. The AMD indicator says *stable* only if, in addition, the AMD cannot bring the innermost
planet to the star.

This cell runs the default check: 10,000 orbits of the innermost planet, the nominal parameters and 20 draws, with WHFast,
a fast symplectic integrator with a fixed step, and a check of the nominal run with IAS15, an adaptive high-accuracy
integrator.

In [ ]:
# About 1.5 minutes in Colab.
!exostab check "TRAPPIST-1" --offline --out exostab-results/a

The numbers behind the Hill verdict, taken from `report.json`, with Δ computed again from the formula above. The figures
show the orbits, and ⟨Y⟩ and the energy error of the nominal run, and the final ⟨Y⟩ of each draw.

In [ ]:
with open("exostab-results/a/report.json") as f:
    report = json.load(f)


def hill_spacing(a1, a2, m1, m2, M_star):
    """Separation in mutual Hill radii, Δ = (a2 - a1) / R_H (lengths in any unit, masses in solar masses)."""
    R_H = ((m1 + m2) / (3 * M_star)) ** (1 / 3) * (a1 + a2) / 2
    return (a2 - a1) / R_H


bodies = report["nominal"]["bodies"]
M_star = report["nominal"]["star_mass_msun"]
pairs = report["indicators"]["hill"]["details"]["pairs"]
print(f"M★ = {M_star} solar masses")
print("Planet   a (AU)    m (solar masses)")
for body in bodies:
    print(f"{body['name'][-1]:6} {body['semi_major_axis_au']:8.5f}   {body['mass_msun']:.3e}")
print()
print("Pair    P₂/P₁   Δ, formula   Δ, report   two-planet limit   needed for 'stable'")
for k, pair in enumerate(pairs):
    b1, b2 = bodies[k], bodies[k + 1]
    spacing = hill_spacing(
        b1["semi_major_axis_au"], b2["semi_major_axis_au"], b1["mass_msun"], b2["mass_msun"], M_star
    )
    ratio = b2["period_days"] / b1["period_days"]
    print(
        f"{b1['name'][-1]} / {b2['name'][-1]}  {ratio:6.3f}  {spacing:10.2f}  {pair['separation_hill']:10.2f}"
        f"  {pair['gladman_critical_hill']:17.2f}  {pair['required_separation_hill']:18.1f}"
    )
print()
print("Software:", report["software"])
display(Image("exostab-results/a/orbits.png", width=480))
display(Image("exostab-results/a/megno.png", width=900))

**Questions (a).**

- **(a1)** Write down the verdicts of Hill spacing, AMD and N-body and the overall verdict. What are ⟨Y⟩ and the
  classification of the nominal run, and how many draws are regular? Why is the N-body *verdict* not the same as the
  classification of the nominal run? Which requirement of the Hill indicator does the system fail, and which pair falls
  furthest short of it? Compute Δ for that pair on paper from the table of a and m, and compare.
- **(a2)** Hill spacing says marginal, and the nominal run is regular. Explain in a few sentences why both can be right.
  Consider which period ratios are close to a simple fraction such as 3:2 or 4:3; whether the inputs of Δ include the
  orbital phases, and whether the inputs of the N-body run do; and the time span behind the rule of Δ ≈ 12 against the
  length of the run on the report card. What does the AMD verdict add?

**Your answers to (a).** Double-click here to write.

- (a1)
- (a2)

## Task (b): where does each indicator change? (about 20 minutes)

HD 127195 has two giant planets, with periods of 534.5 and 837 days, minimum masses of 213 and 207 Earth masses and
published eccentricities of 0.04 and 0.05. Its orbital phases are not published. exostab draws them from the seed and
holds them fixed through a sweep.

**These are what-if runs.** `exostab sweep` gives planet b eccentricities from 0 to 0.4 and keeps every other parameter at
its nominal value. Each value of e_b is one run of 10,000 orbits, with no draws. In the N-body column, *unstable* means that
the run ended in a close encounter or escape. The rows describe these parameter sets, not the real HD 127195.

In [ ]:
# About 15 seconds in Colab.
!exostab sweep "HD 127195" --planet b --param e --range 0 0.4 --steps 21 --offline --out exostab-results/b
display(Image("exostab-results/b/sweep.png", width=700))

The numbers behind that table, from `sweep.csv`: the spacing Δ, the margin of the Hill test (explained below), the largest
AMD ratio β and how each run ended.

In [ ]:
with open("exostab-results/b/sweep.csv") as f:
    sweep = list(csv.DictReader(f))

print(" e_b      Δ    margin        β   end of run")
for row in sweep:
    print(
        f"{float(row['value']):4.2f}  {float(row['hill_min_separation']):5.3f}  {float(row['hill_margin_min']):+8.5f}"
        f"  {float(row['amd_beta_max']):7.3f}   {row['status']} at {float(row['t_end_yr']):,.0f} yr"
    )

The formula for Δ has no e in it. The test behind the Hill column is the eccentric form of the criterion of Gladman
(1993), as restated by Veras & Armitage (2004) and Giuppone et al. (2013). With μᵢ = mᵢ/M★, α = μ₁ + μ₂,
γᵢ = √(1 − eᵢ²) and δ = √(a₂/a₁), a pair is Hill stable if

α⁻³ (μ₁ + μ₂/δ²) (μ₁γ₁ + μ₂γ₂δ)² > 1 + 3^(4/3) μ₁μ₂ / α^(4/3).

The *margin* is the left side divided by the right side, minus 1. Expanding both sides for small masses, small
eccentricities and a small gap between the orbits gives a form that can be read directly, with e_H = (α/3)^(1/3):

Δ² > 12 + (4/3) [(α/μ₂) e₁² + (α/μ₁) e₂²] / e_H²,  which for equal masses is  Δ² > 12 + (8/3) (e₁² + e₂²) / e_H².

The next cell writes these out and compares them with the sweep.

In [ ]:
MEARTH_IN_MSUN = 3.003e-6


def gladman_margin(m1, m2, a1, a2, e1, e2, M_star):
    """Left side / right side - 1 of the criterion above: positive means Hill stable."""
    mu1, mu2 = m1 / M_star, m2 / M_star
    alpha = mu1 + mu2
    delta = (a2 / a1) ** 0.5
    gamma1, gamma2 = (1 - e1**2) ** 0.5, (1 - e2**2) ** 0.5
    left = alpha**-3 * (mu1 + mu2 / delta**2) * (mu1 * gamma1 + mu2 * gamma2 * delta) ** 2
    right = 1 + 3 ** (4 / 3) * mu1 * mu2 / alpha ** (4 / 3)
    return left / right - 1


# The nominal values of HD 127195, as in the archive's row.
M_star = 1.34
m_b, m_c = 212.945 * MEARTH_IN_MSUN, 206.588 * MEARTH_IN_MSUN
P_b, P_c = 534.5 / 365.25, 836.99 / 365.25
e_c = 0.05
# Kepler's third law in AU, years and solar masses: a³ = M P², where M is the mass within the orbit.
a_b = ((M_star + m_b) * P_b**2) ** (1 / 3)
a_c = ((M_star + m_b + m_c) * P_c**2) ** (1 / 3)

spacing = hill_spacing(a_b, a_c, m_b, m_c, M_star)
row = next(r for r in sweep if float(r["value"]) == 0.1)
margin = gladman_margin(m_b, m_c, a_b, a_c, 0.1, e_c, M_star)
print(f"a_b = {a_b:.4f} AU, a_c = {a_c:.4f} AU")
print(f"Δ from the formula: {spacing:.3f}; sweep: {float(row['hill_min_separation']):.3f}")
print(f"Margin at e_b = 0.1 from the formula: {margin:+.5f}; sweep: {float(row['hill_margin_min']):+.5f}")

e_crit = next(e / 1000 for e in range(401) if gladman_margin(m_b, m_c, a_b, a_c, e / 1000, e_c, M_star) <= 0)
e_hill = ((m_b + m_c) / (3 * M_star)) ** (1 / 3)
e_estimate = ((spacing**2 - 12) * (3 / 8) * e_hill**2 - e_c**2) ** 0.5
e_touch = a_c * (1 - e_c) / a_b - 1
print(f"The margin reaches 0 at e_b = {e_crit:.3f}")
print(f"Estimate from the equal-mass expansion: e_b = {e_estimate:.3f} (e_H = {e_hill:.4f})")
print(f"The radial ranges of the two orbits touch, a_b (1 + e_b) = a_c (1 - e_c), at e_b = {e_touch:.3f}")

The sweep used the random phases of seed 0. This cell repeats it with seed 2, which gives other random phases; a, m and e
stay as they were. These are still what-if runs.

In [ ]:
# About 15 seconds in Colab.
!exostab sweep "HD 127195" --planet b --param e --range 0 0.4 --steps 21 --offline --seed 2 --out exostab-results/b-seed2 > /dev/null
with open("exostab-results/b-seed2/sweep.csv") as f:
    other = list(csv.DictReader(f))

print(" e_b   N-body, seed 0   N-body, seed 2   Hill and AMD the same")
for k, row in enumerate(sweep):
    same = row["hill_verdict"] == other[k]["hill_verdict"] and row["amd_verdict"] == other[k]["amd_verdict"]
    print(f"{float(row['value']):4.2f}   {row['nbody_class']:14}   {other[k]['nbody_class']:14}   {same}")

**Questions (b).**

- **(b1)** Between which two values of e_b does the AMD verdict first change, and the Hill verdict? Where does the first run
  turn chaotic, and where is the first close encounter? Which indicator changes more than once, in both directions?
- **(b2)** Δ does not change at all. Explain this from the formula for Δ, and say where e enters the Hill verdict. From the
  expanded form, how should the margin depend on e_b when e_b is small? Does the middle panel of the figure agree? Compare
  the e_b where the margin reaches 0 with the e_b where the radial ranges touch and the e_b of the first close encounter.
  For e_b from 0.12 to 0.14 the Hill column says *unstable* and the runs are regular. Is that a contradiction?
- **(b3)** What changes with the seed, what does not, and why? What do the two seeds say at the published e_b = 0.04? What
  does that tell you about reading one row of the N-body column as an answer for the real planets?

**Your answers to (b).** Double-click here to write.

- (b1)
- (b2)
- (b3)

## Task (c): the horizon (about 20 minutes)

The default run of TRAPPIST-1 lasts 10,000 orbits of planet b, about 41 years. In this task only the length of the run
changes: the parameters are those of the archive's row, so these are not what-if runs. MEGNO ⟨Y⟩ follows how fast a small
difference from the trajectory grows, starting from a random direction. For regular (quasi-periodic) motion ⟨Y⟩ settles
at 2, and for chaotic motion it keeps growing, about in proportion to time. Over a short run ⟨Y⟩ has not settled, so a
regular run can end well below 2.

The first run is five times longer, with no draws. The right panel of the figure stays empty for that reason.

In [ ]:
# About 3 minutes in Colab: the IAS15 check takes nearly all of it.
!exostab check "TRAPPIST-1" --offline --orbits 50000 --draws 0 --out exostab-results/c
display(Image("exostab-results/c/megno.png", width=900))

The next cell runs several lengths without the IAS15 check and makes a table from the reports. Some rows have 10 draws, the
first 10 of task (a); *r/c/e* counts them as regular, chaotic, or ended by a close encounter or escape. λ is the largest
Lyapunov exponent of the nominal run, and 1/λ the Lyapunov time: the time in which a small difference between two nearby
trajectories grows by a factor e. REBOUND's `lyapunov()` gives the slope of ⟨Y⟩ against time. For chaotic motion ⟨Y⟩ grows
as λt/2, so the table shows λ as twice that slope. The last line repeats the default length with seed 3. For TRAPPIST-1
the nominal values do not depend on the seed: only the random starting direction of MEGNO changes.

In [ ]:
# About 3.5 minutes in Colab.
print("  orbits  years    <Y>  nominal   λ (1/yr)  1/λ (yr)  r/c/e     Hill      AMD     N-body    overall")
for orbits, draws in ((10000, 10), (30000, 0), (50000, 10), (100000, 0), (200000, 0)):
    out = f"exostab-results/c-{orbits}"
    !exostab check "TRAPPIST-1" --offline --orbits {orbits} --draws {draws} --no-ias15 --no-plots --out {out} > /dev/null
    with open(f"{out}/report.json") as f:
        r = json.load(f)
    nbody = r["indicators"]["nbody"]
    nominal = nbody["details"]["nominal"]
    # REBOUND's lyapunov() is the slope of <Y> against time, and <Y> grows as λt/2.
    lam = 2 * nominal["lyapunov_per_yr"]
    lyapunov_time = f"{1 / lam:.0f}" if lam > 0 else "-"
    counts = nbody["details"]["classification_counts"]
    found = f"{counts['regular']}/{counts['chaotic']}/{counts['unstable']}" if draws else "no draws"
    print(
        f"{orbits:8,} {nominal['t_end_yr']:6.0f} {nominal['megno']:6.2f}  {nominal['classification']:8} {lam:9.4f}"
        f" {lyapunov_time:>9}  {found:8}  {r['indicators']['hill']['verdict']:8}  {r['indicators']['amd']['verdict']:6}"
        f"  {nbody['verdict']:8}  {r['overall']['verdict']}"
    )
print(f"Last run: {nominal['n_orbits_innermost']:,.0f} orbits in {nominal['walltime_s']:.1f} s of computing.")

!exostab check "TRAPPIST-1" --offline --draws 0 --no-ias15 --no-plots --seed 3 --out exostab-results/c-seed3 > /dev/null
with open("exostab-results/c-seed3/report.json") as f:
    nominal = json.load(f)["indicators"]["nbody"]["details"]["nominal"]
print(f"10,000 orbits with seed 3: <Y> = {nominal['megno']:.2f}, {nominal['classification']}.")

**Questions (c).**

- **(c1)** In the figure, at about what time does ⟨Y⟩ pass 2.2 for good? Compare that with the length of the default run.
  Until about what time do WHFast and IAS15 agree? The lower panel shows that both keep the energy very well. Why can two
  accurate integrators still end with different ⟨Y⟩?
- **(c2)** From the table: what changes as the run gets longer, and what does not? Why can the Hill and AMD verdicts not
  change? Does the overall verdict change, and what changes behind it? Estimate λ from the ⟨Y⟩ of the two longest runs, with
  ⟨Y⟩ ≈ λt/2, and compare it with the table. How many Lyapunov times does the default run cover, and what does that say about
  λ in the short runs? What does "chaotic" say about these runs, and what does it not say?
- **(c3)** From the speed of the last run, estimate how long one run of a billion years would take on this computer
  (a billion years is about 2.4 × 10¹¹ orbits of planet b). Then say in two or three sentences what no finite run can show,
  however long it is. Use a row of the table as your example. The last line changes only the seed: what does it say about
  the label in the first row?

**Your answers to (c).** Double-click here to write.

- (c1)
- (c2)
- (c3)

## Task (d): what a stable verdict rests on (about 20 minutes)

Kepler-36 has two planets, with periods of 13.9 and 16.2 days. In the catalog of 26 October 2026 it has the smallest
spacing of any system that exostab calls stable ([What the catalog shows](https://mimitoz62.github.io/exostab/findings/)).
The archive's row gives both planets e = 0 without an uncertainty, from Vissapragada et al. (2020). That paper fixes e = 0
only in its light-curve fit; its dynamical fit gives eccentricity-vector components of up to about 0.05. exostab treats
such a value as assumed, not measured. This cell uses the catalog's number of draws, 10.

In [ ]:
# About 20 seconds in Colab.
!exostab check "Kepler-36" --offline --draws 10 --out exostab-results/d

The same system as a YAML file, with the values of the archive's row in the snapshot. A planet with no `eccentricity` line
gets an assumed e = 0, and a planet with no `omega_deg` line a random argument of periastron in every run, as in the archive
run. Before you change an input, check that the file gives the archive run's result. In the table below, margin and β are
those of tasks (a) and (b), and r/c/e counts the draws as in task (c).

In [ ]:
%%writefile exostab-results/kepler36.yaml
# Kepler-36 with the values of the archive's row in the snapshot of 28 September 2026.
# With an eccentricity line this is a what-if: to try one, remove the # and set a value.
name: Kepler-36 (exercise file)
notes: What-if file of the exostab exercise, not an archive record.
star:
  name: Kepler-36
  mass_msun: {value: 1.034, err: 0.022}
planets:
  - name: Kepler-36 b
    period_days: {value: 13.86825, err: 0.0005}
    mass_earth: {value: 3.83, err_plus: 0.11, err_minus: 0.1}
    # eccentricity: 0.02
    transit_time_jd: {value: 2454974.6491, err: 0.0032}
  - name: Kepler-36 c
    period_days: {value: 16.21865, err: 0.0001}
    mass_earth: {value: 7.13, err: 0.18}
    # eccentricity: 0.02
    transit_time_jd: {value: 2455004.677414, err: 0.000655}

In [ ]:
# About 20 seconds in Colab. To try the file with other values, edit the cell above and run both cells again.
!exostab check --from-file exostab-results/kepler36.yaml --draws 10 --out exostab-results/d-file > /dev/null

HEADER = "Run                    Hill     AMD      N-body   overall  margin        β  nominal      <Y>    ended     r/c/e"


def read_report(out):
    """The report of a check."""
    with open(f"{out}/report.json") as f:
        return json.load(f)


def print_row(label, r):
    """One line of a table: the verdicts, the Hill margin, β, the nominal run and the draws."""
    hill, amd, nbody = r["indicators"]["hill"], r["indicators"]["amd"], r["indicators"]["nbody"]
    nominal, counts = nbody["details"]["nominal"], nbody["details"]["classification_counts"]
    margin = hill["details"]["pairs"][0]["gladman_margin"]
    beta = max(p["beta"] for p in amd["details"]["pairs"])
    stopped = nominal["classification"] == "unstable"
    print(
        f"{label:22} {hill['verdict']:9}{amd['verdict']:9}{nbody['verdict']:9}{r['overall']['verdict']:9}"
        f"{margin:+.5f} {beta:6.3f}  {nominal['status'] if stopped else nominal['classification']:9}"
        f" {'-' if stopped else format(nominal['megno'], '.2f'):>6} {nominal['t_end_yr']:6.3g} yr"
        f"  {counts['regular']}/{counts['chaotic']}/{counts['unstable']}"
    )


print(HEADER)
print_row("Archive row", read_report("exostab-results/d"))
print_row("File", read_report("exostab-results/d-file"))

**What-if runs, not results for Kepler-36.** This cell copies the file, sets the same e for both planets, holds it fixed in
the draws and runs each copy with the same settings: 10,000 orbits and 10 draws. ω stays random. The last line uses the
expansion from task (b) to estimate where the Hill test should change.

In [ ]:
# About 1.5 minutes in Colab.
with open("exostab-results/kepler36.yaml") as f:
    base = yaml.safe_load(f)


def write_what_if(path, label, e_b, e_c):
    """A copy of the file with other eccentricities, labelled as a what-if."""
    system = copy.deepcopy(base)
    system["name"] = f"Kepler-36 what-if: {label}"
    system["planets"][0]["eccentricity"] = e_b
    system["planets"][1]["eccentricity"] = e_c
    with open(path, "w") as f:
        yaml.safe_dump(system, f)


print(HEADER)
print_row("Archive row, e assumed", read_report("exostab-results/d"))
for e in (0.005, 0.01, 0.02, 0.03, 0.04):
    file = f"exostab-results/kepler36-whatif-e{e}.yaml"
    out = f"exostab-results/d-e{e}"
    write_what_if(file, f"e = {e} for both planets", e, e)
    !exostab check --from-file {file} --draws 10 --no-plots --out {out} > /dev/null
    print_row(f"What-if: e = {e}", read_report(out))

m_b, m_c = 3.83 * MEARTH_IN_MSUN, 7.13 * MEARTH_IN_MSUN
M_star = 1.034
spacing = read_report("exostab-results/d")["indicators"]["hill"]["details"]["pairs"][0]["separation_hill"]
e_hill = ((m_b + m_c) / (3 * M_star)) ** (1 / 3)
weight = (m_b + m_c) / m_c + (m_b + m_c) / m_b
e_estimate = ((spacing**2 - 12) * (3 / 4) * e_hill**2 / weight) ** 0.5
print(f"Δ = {spacing:.2f}. Expansion, same e for both planets: margin 0 at e = {e_estimate:.3f}")

**What-if, not a result for Kepler-36.** Carter et al. (2012) give upper limits for these eccentricities: 0.039 for b and
0.033 for c (the archive's `ps` table). Here the file states them as limits: the nominal run takes each limit, and the draws
are uniform below it. Only the seed changes between the rows, so only the random draws change, ω among them.

In [ ]:
# About 40 seconds in Colab.
limit_b = {"value": 0.039, "upper_limit": True}
limit_c = {"value": 0.033, "upper_limit": True}
print(HEADER)
for seed in (0, 1, 2):
    file = f"exostab-results/kepler36-whatif-limits-seed{seed}.yaml"
    out = f"exostab-results/d-limits-seed{seed}"
    write_what_if(file, "the upper limits of Carter et al. 2012", limit_b, limit_c)
    !exostab check --from-file {file} --draws 10 --seed {seed} --no-plots --out {out} > /dev/null
    print_row(f"What-if: limits seed {seed}", read_report(out))

**Questions (d).**

- **(d1)** On the report card of the archive run, which values are marked as assumed or random? Read the text under AMD. What
  is β, and why does the AMD test have nothing to test here? Does the file give the result of the archive run?
- **(d2)** In the what-if table, at which e does each indicator first differ from the archive run? Which reacts first? Compare
  the Hill column with the estimate from the expansion. With the limits, what stays the same across the three seeds, and what
  does not? What part of the input does that point to?
- **(d3)** Deck et al. (2012) integrated initial conditions consistent with the observations and found chaotic orbits with a
  Lyapunov time of about 10 years; only about 4.5% of their sample showed no large-scale instabilities over about 200 million
  years. Does the regular run with e = 0 contradict that? In two or three sentences: what do these what-if runs show about the
  input? In two or three more: what do they not show about the planets?

**Your answers to (d).** Double-click here to write.

- (d1)
- (d2)
- (d3)

## To finish (about 5 minutes)

Someone writes in a report: "exostab says that Kepler-36 is stable and TRAPPIST-1 is not." Rewrite that sentence so that it
says only what these runs support. Name the settings and what the verdicts rest on.

**Your answer.** Double-click here to write.

## Further reading

- [Methods](https://mimitoz62.github.io/exostab/methods/), with the formulas and references behind each indicator.
- [Data and assumptions](https://mimitoz62.github.io/exostab/data-assumptions/), with the rules that fill the gaps in the
  archive's data.
- [What the catalog shows](https://mimitoz62.github.io/exostab/findings/), with the catalog of 26 October 2026 and the facts
  about Kepler-36 used in task (d), with their references.
- [CLI reference](https://mimitoz62.github.io/exostab/cli/), with every option of `exostab check` and `exostab sweep`.
- If you use the results beyond this exercise, please
  [cite exostab, the methods and the archive](https://mimitoz62.github.io/exostab/citing/).